In [5]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

ModuleNotFoundError: No module named 'requests'

In [3]:
headers = {
    'authority': 'books.toscrape.com',
    'user-agent': 'Mozilla/5.0'
}

In [ ]:
base_url = 'http://books.toscrape.com/catalogue/page-{}.html'

In [ ]:
num_pages = 5
RATING_MAP = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}

In [ ]:
def booksHtml(base_url, num_pages):

    soups = []

    for page_no in range(1, num_pages + 1):

        url = base_url.format(page_no)

        response = requests.get(url, headers=headers)

        soup = BeautifulSoup(response.text, 'lxml')

        soups.append(soup)

        print(f'Page {page_no} fetched (status {response.status_code})')

    return soups

In [ ]:
def getBooks(html_data):

    data_dicts = []

    boxes = html_data.select('article.product_pod')

    for box in boxes:

        title        = box.select_one('h3 a')['title']
        price        = box.select_one('.price_color').text.strip()
        rating_word  = box.select_one('p.star-rating')['class'][1]
        rating       = RATING_MAP.get(rating_word, 'N/A')
        availability = box.select_one('.availability').text.strip()
        link         = 'http://books.toscrape.com/catalogue/' + box.select_one('h3 a')['href']

        data_dicts.append({
            'Title'        : title,
            'Price'        : price,
            'Rating'       : rating,
            'Availability' : availability,
            'Link'         : link
        })

    return data_dicts

In [ ]:
html_datas = booksHtml(base_url, num_pages)
books = []
for html_data in html_datas:
    book = getBooks(html_data)
    books += book

In [ ]:
df_books = pd.DataFrame(books)
df_books

In [ ]:
df_books.to_csv('books.csv', index=False)